In [2]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

print(es.info())

{'name': 'c88a00953813', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'PJOgfzDXRyO8ycBAszDlag', 'version': {'number': '8.4.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '42f05b9372a9a4a470db3b52817899b99a76ee73', 'build_date': '2022-10-04T07:17:24.662462378Z', 'build_snapshot': False, 'lucene_version': '9.3.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [3]:
import requests

def load_faq_data():
    docs_url = 'https://datatalks.club/faq/json/courses.json'
    response = requests.get(docs_url)
    courses_raw = response.json()

    documents = []
    url_prefix = 'https://datatalks.club/faq'

    for course in courses_raw:
        course_url = f'{url_prefix}{course["path"]}'
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()

        documents.extend(course_data)

    return documents

In [4]:
documents = load_faq_data()

In [5]:
len(documents)

1208

In [6]:
type(documents)

list

In [7]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [8]:
mapping = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "section": {"type": "text"},
            "question": {"type": "text"},
            "answer": {"type": "text"},
            "course": {"type": "keyword"},
        }
    }
}

INDEX_NAME = "course-qa"


In [9]:
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

es.indices.create(index=INDEX_NAME, body=mapping)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course-qa'})

In [10]:
es.indices.get(index=INDEX_NAME)

ObjectApiResponse({'course-qa': {'aliases': {}, 'mappings': {'properties': {'answer': {'type': 'text'}, 'course': {'type': 'keyword'}, 'question': {'type': 'text'}, 'section': {'type': 'text'}}}, 'settings': {'index': {'routing': {'allocation': {'include': {'_tier_preference': 'data_content'}}}, 'number_of_shards': '1', 'provided_name': 'course-qa', 'creation_date': '1778867363234', 'number_of_replicas': '0', 'uuid': 'WnWHnfAnS9axhAcB8igqjw', 'version': {'created': '8040399'}}}}})

In [11]:
from tqdm import tqdm

In [12]:
for doc in tqdm(documents):
    es.index(
        index=INDEX_NAME,
        # id=doc["id"],
        document=doc
    )

100%|██████████| 1208/1208 [00:06<00:00, 179.06it/s]


In [13]:
es.indices.refresh(index=INDEX_NAME)

ObjectApiResponse({'_shards': {'total': 1, 'successful': 1, 'failed': 0}})

In [37]:
def search_elastic(query,course):

    # search_query = {
    #     "size": 5,
    #     "query": {
    #         "bool": {
    #             "must": {
    #                 "multi_match": {
    #                     "query": query,
    #                     "fields": ["question^3", "section^0.5"],
    #                     "type": "best_fields"
    #                 }
    #             },
    #             "filter": {
    #                 "term": {
    #                     "course": "mlops-zoomcamp"
    #                 }
    #             }
                
    #         }
            
    #     }
    # }

    search_query =  {
        "size": 5,
        "query": {
            "bool": {
                "must": [
                    {
                    "multi_match": {
                        "query": query,
                        "fields": [
                            "question^3.0",
                            "section^0.5",
                            "answer^3.0"
                        ],
                        "type": "best_fields"
                    }
                    }
                ],
                "filter": [
                    {
                    "term": {
                        "course": course
                    }
                    }
                ]
            }
        }
    }

    res = es.search(index=INDEX_NAME, body=search_query)

    result_docs = []
    for hit in res["hits"]["hits"]:
        # print(hit["_score"])
        # print(hit["_source"]["question"])
        # print("-" * 50)
        result_docs.append(hit['_source'])
    
    return result_docs 

In [38]:
query = "how do I run docker?"
course = "mlops-zoomcamp" #"data-engineering-zoomcamp"
search_elastic(query, course)

[{'id': '884bb9afb7',
  'course': 'mlops-zoomcamp',
  'section': 'Module 4: Deployment',
  'question': 'Docker: Passing envs to my docker image',
  'answer': '**Problem Description:** \n\nI was having issues because my Python script was not reading AWS credentials from environment variables. After building the image, I was running it like this:\n\n```bash\ndocker run -it homework-04 -e AWS_ACCESS_KEY_ID=xxxxxxxx -e AWS_SECRET_ACCESS_KEY=xxxxxx\n```\n\n1. **Environment Variables Order: **\n   \n   You can set environment variables like `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, and `AWS_SESSION_TOKEN` (if using AWS STS). Ensure these variables are passed before the image name:\n\n   ```bash\n   docker run -e AWS_ACCESS_KEY_ID=xxxxxxxx -e AWS_SECRET_ACCESS_KEY=xxxxxx -it homework-04\n   ```\n\n2. **Using an Env File:**\n\n   You can pass an env file by using the following command, assuming your env file is named `.env`:\n\n   ```bash\n   docker run -it homework-04 --env-file .env\n   

In [17]:
from dotenv import load_dotenv
from openai import OpenAI
import os

# Load .env file
load_dotenv()

# Read API key
api_key = os.getenv("OPENAI_API_KEY")

# Initialize client
client = OpenAI(api_key=api_key)

# Example request
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": "Hello!"}
    ]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [18]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [19]:
PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

def build_prompt(query, search_results):
    context = build_context(search_results)
    return PROMPT_TEMPLATE.format(question=query, context=context)

In [20]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'):
    input_messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = client.responses.create(
        model=model,
        input=input_messages
    )

    return response.output_text

In [40]:
INSTRUCTIONS = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

def rag(query, model='gpt-5.4-mini'):
    search_results = search_elastic(query,course)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [41]:
query = "How do I run Docker?"

In [42]:
answer = rag(query)

In [43]:
print(answer)

To run a Docker image, use:

```bash
docker run -it --rm mlops-learn
```

If you need to pass AWS environment variables, make sure they come **before** the image name:

```bash
docker run -e AWS_ACCESS_KEY_ID=xxxxxxxx -e AWS_SECRET_ACCESS_KEY=xxxxxx -it homework-04
```

You can also use an env file:

```bash
docker run -it homework-04 --env-file .env
```

Or mount AWS credentials from your host:

```bash
docker run -it --rm -v ~/.aws:/root/.aws homework:v1
```
